# Unit 4. Final Project - Create, Test, and Certify Your Agent

### Preguntas con herramientas

In [1]:
tasks = [
    {"task_id": "demo-01", "operation": "multiply", "arguments": {"a": 6, "b": 7}},
    {"task_id": "demo-02", "operation": "read_field", "arguments": {"field": "city"}},
    {"task_id": "demo-03", "operation": "read_field", "arguments": {"field": "date"}},
]
references = {"demo-01": "42", "demo-02": "Bogotá", "demo-03": "2026-01-15"}
document = {"city": "Bogotá"}

def multiply(a, b):
    return a * b

def read_field(field):
    return document.get(field, "")

def answer(task):
    tools = {"multiply": multiply, "read_field": read_field}
    value = tools[task["operation"]](**task["arguments"])
    return str(value)

answers = [
    {"task_id": task["task_id"], "submitted_answer": answer(task)}
    for task in tasks
]
assert [item["submitted_answer"] for item in answers] == ["42", "Bogotá", ""]
for item in answers:
    print(f"{item['task_id']} → {item['submitted_answer'] or '(sin respuesta)'}")


demo-01 → 42
demo-02 → Bogotá
demo-03 → (sin respuesta)


### Puntuación por coincidencia exacta

In [2]:
def exact_match(answers, references):
    predicted = {item["task_id"]: item["submitted_answer"] for item in answers}
    if not references or len(predicted) != len(answers) or predicted.keys() != references.keys():
        raise ValueError("Se requiere exactamente una respuesta por tarea de referencia.")
    return {
        task_id: predicted[task_id].strip() == expected.strip()
        for task_id, expected in references.items()
    }

matches = exact_match(answers, references)
assert matches == {"demo-01": True, "demo-02": True, "demo-03": False}
assert exact_match([{"task_id": "x", "submitted_answer": " 42 "}], {"x": "42"})["x"]
correct = sum(matches.values())
score = correct / len(matches)
assert score == 2 / 3
for task_id, is_correct in matches.items():
    print(f"{task_id}: {'correcta' if is_correct else 'incorrecta'}")
print(f"Resultado sobre preguntas inventadas: {correct}/{len(matches)} ({score:.1%})")


demo-01: correcta
demo-02: correcta
demo-03: incorrecta
Resultado sobre preguntas inventadas: 2/3 (66.7%)


### Formato de respuestas para la plantilla

In [3]:
import json

def build_submission(username, agent_code, answers):
    return {"username": username, "agent_code": agent_code, "answers": answers}

demo_submission = build_submission(
    username="usuario-de-ejemplo",
    agent_code="https://huggingface.co/spaces/usuario-de-ejemplo/agente/tree/main",
    answers=answers,
)
payload = json.dumps(demo_submission, ensure_ascii=False, indent=2)
assert json.loads(payload)["answers"] == answers
print(payload)


{
  "username": "usuario-de-ejemplo",
  "agent_code": "https://huggingface.co/spaces/usuario-de-ejemplo/agente/tree/main",
  "answers": [
    {
      "task_id": "demo-01",
      "submitted_answer": "42"
    },
    {
      "task_id": "demo-02",
      "submitted_answer": "Bogotá"
    },
    {
      "task_id": "demo-03",
      "submitted_answer": ""
    }
  ]
}
